
## CELEBAL TECHNOLOGIES

#### Data Engineering Internship (CEI Program 2026)

### Week 5 Assignment

#### Apache Spark: DataFrames, Transformations & Data Cleaning

---

### Submitted By
**Harshit Sharma**  
*Data Engineering Intern*

---


### Q1. Key Limitations of Traditional MapReduce That Make Spark Preferred:-
MapReduce was revolutionary when it was introduced, but it has some fundamental design choices 
that make it very slow for modern workloads — especially when we need to run iterative algorithms or 
interactive queries. Here's what makes Spark a better fit: 

| Limitation in MapReduce | How Spark Addresses It |
|-------------------------|-------------------------|
| Every intermediate result is written to HDFS (disk I/O is the bottleneck) | Spark keeps intermediate data in memory (RAM) using RDDs/DataFrames, drastically reducing I/O |
| A job is locked into two phases: Map → Reduce. Complex pipelines require multiple chained jobs | Spark supports a rich DAG of transformations (filter, join, aggregate, join, etc.) within a single job |
| No native support for streaming. Separate tools such as Storm are required | Spark Streaming and Structured Streaming handle real-time data using the same API |
| Each MapReduce iteration re-reads data from disk, making machine learning training loops inefficient | Spark can cache datasets in memory, making repeated computations 10–100x faster |
| High startup latency for each job, making exploratory analysis difficult | Spark provides interactive environments such as `spark-shell` and PySpark REPL for near real-time feedback |
| Writing MapReduce applications requires verbose Java code, even for simple tasks | Spark offers concise DataFrame and SQL APIs that are easier to write and maintain |

In practice, when training a machine learning model that needs 100 iterations over the same dataset, 
MapReduce would read from disk 100 times. Spark reads the data once, caches it, and iterates entirely 
in memory. That is the core reason companies like Uber, Netflix, and Airbnb migrated from Hadoop 
MapReduce to Spark. 


## Q2. How Spark Uses In-Memory Computing to Speed Up Iterative ML Algorithms

To understand Spark's advantage, consider training a machine learning model such as **Logistic Regression** that requires multiple iterations (epochs) over the same dataset.

### Disk-Based System (MapReduce)

In MapReduce, each iteration performs the following steps:

* Read training data from HDFS
* Execute the Map phase
* Write intermediate results to disk
* Execute the Reduce phase
* Write output back to HDFS

For every new iteration, the entire process repeats.

**Example:**

* Iteration 1: Read → Map → Write → Reduce → Write
* Iteration 2: Read → Map → Write → Reduce → Write
* ...
* Iteration 50: Read → Map → Write → Reduce → Write

As a result, 50 iterations require 50 full disk reads and 50 writes. Network transfer and disk I/O become the primary performance bottlenecks.

### Spark In-Memory Approach

Spark improves performance by keeping frequently used data in memory.

The workflow is:

* Read training data once from storage
* Cache the DataFrame using `.cache()` or `.persist()`
* First iteration loads data into memory
* Subsequent iterations reuse the cached data directly from RAM
* Disk access is needed only for the initial load and final output

### How Caching Works

Spark represents computations as a **Directed Acyclic Graph (DAG)**. When `.cache()` is invoked, Spark stores DataFrame partitions in executor memory. Future actions on the same DataFrame reuse the cached partitions instead of recomputing transformations or reading data from disk.



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg

spark = SparkSession.builder.appName("MLExample").getOrCreate()

# Create sample training dataset
training_df = spark.range(1, 1000001).toDF("feature")

# Simulate multiple ML iterations
for epoch in range(1, 6):
    avg_feature = training_df.select(avg("feature")).collect()[0][0]
    print(f"Epoch {epoch}: Average Feature = {avg_feature}")

Epoch 1: Average Feature = 500000.5
Epoch 2: Average Feature = 500000.5
Epoch 3: Average Feature = 500000.5
Epoch 4: Average Feature = 500000.5
Epoch 5: Average Feature = 500000.5


**Conclusion:** Because Spark keeps frequently accessed datasets in memory, it can be significantly faster than MapReduce for iterative machine learning workloads.

## Q3. Removing Duplicate Rows Based on Specific Columns 

In [0]:

from pyspark.sql import SparkSession 
 
spark = SparkSession.builder.appName('DeduplicationExample').getOrCreate() 
 
# Sample DataFrame with intentional duplicates 
data = [ 
    (1, '2024-01-15', 'Amazon', 250.00), 
    (1, '2024-01-15', 'Amazon', 250.00),   # exact duplicate 
    (1, '2024-01-15', 'Flipkart', 100.00), # same user+date, diff merchant 
    (2, '2024-01-16', 'Amazon', 300.00), 
] 
columns = ['user_id', 'transaction_date', 'merchant', 'amount'] 
df = spark.createDataFrame(data, columns) 
 
# Drop duplicates based on user_id and transaction_date only 
# Keeps the FIRST occurrence of each (user_id, transaction_date) pair 
df_deduped = df.dropDuplicates(['user_id', 'transaction_date']) 
 
df_deduped.show() 

+-------+----------------+--------+------+
|user_id|transaction_date|merchant|amount|
+-------+----------------+--------+------+
|      1|      2024-01-15|  Amazon| 250.0|
|      2|      2024-01-16|  Amazon| 300.0|
+-------+----------------+--------+------+



*Note*: .dropDuplicates() without any arguments removes only rows that are completely identical across 
every column. Passing a subset list makes the deduplication logic explicit and intentional — which is 
almost always what you want in a data pipeline.

%md

## Q4. Filter by Region and Group By Product Category to Find Average Sale Amount

This task involves two transformations:

1. Filter the dataset to include only records from the **West** region.
2. Group the filtered records by **product_category** and calculate the average sale amount.

Apache Spark provides multiple ways to achieve this. The two most common approaches are:

* **DataFrame API** (preferred in ETL and production pipelines)
* **Spark SQL** (easy to read and write for SQL users)

Both approaches produce the same result.



In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    ("West", "Electronics", 1200),
    ("West", "Electronics", 1500),
    ("West", "Furniture", 800),
    ("West", "Furniture", 1000),
    ("West", "Clothing", 500),
    ("East", "Electronics", 1300),
    ("South", "Furniture", 900)
]

columns = ["region", "product_category", "sale_amount"]

df_sales = spark.createDataFrame(data, columns)

df_sales.show()

+------+----------------+-----------+
|region|product_category|sale_amount|
+------+----------------+-----------+
|  West|     Electronics|       1200|
|  West|     Electronics|       1500|
|  West|       Furniture|        800|
|  West|       Furniture|       1000|
|  West|        Clothing|        500|
|  East|     Electronics|       1300|
| South|       Furniture|        900|
+------+----------------+-----------+



In [0]:
from pyspark.sql import functions as F

result = (
    df_sales
    .filter(F.col("region") == "West")
    .groupBy("product_category")
    .agg(F.avg("sale_amount").alias("avg_sale"))
    .orderBy(F.desc("avg_sale"))
)

result.show()

+----------------+--------+
|product_category|avg_sale|
+----------------+--------+
|     Electronics|  1350.0|
|       Furniture|   900.0|
|        Clothing|   500.0|
+----------------+--------+



### Why This Transformation Is Useful

Businesses often need to analyze regional sales performance. By filtering a specific region and aggregating sales by product category, organizations can identify which product categories generate the highest average revenue within that region.

The resulting dataset can support:

* Sales performance analysis
* Product strategy decisions
* Regional business reporting
* Dashboard development


%md

## Q5. Difference Between `.na.drop()` and `.na.fill()` with Code Example

Missing or null values are common in real-world datasets. Apache Spark provides multiple methods to handle null values, with `.na.drop()` and `.na.fill()` being the most commonly used.

### Comparison

| Method       | What It Does                               | Use Case                                        |
| ------------ | ------------------------------------------ | ----------------------------------------------- |
| `.na.drop()` | Removes rows containing null values        | When records with missing values are unusable   |
| `.na.fill()` | Replaces null values with specified values | When a reasonable default value can be assigned |

### When to Use Each Method

#### `.na.drop()`

Use this method when the missing value is critical and retaining the row could affect data quality.

**Example:**

* Customer ID is missing
* Transaction Date is missing
* Primary Key is null

In such cases, dropping the row is often the safest choice.

#### `.na.fill()`

Use this method when a sensible replacement value exists.

**Examples:**

* Missing city → `"Unknown"`
* Missing salary → `0`
* Missing rating → `-1`

This helps preserve records while handling incomplete data.


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Sample data containing null values
data = [
    (1, "Alice", "Active"),
    (2, "Bob", None),
    (3, None, "Inactive"),
    (4, "Diana", None)
]

columns = ["id", "name", "status"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

Original Data:
+---+-----+--------+
| id| name|  status|
+---+-----+--------+
|  1|Alice|  Active|
|  2|  Bob|    NULL|
|  3| NULL|Inactive|
|  4|Diana|    NULL|
+---+-----+--------+



In [0]:
# Fill null values in status column

df_filled = df.na.fill({
    "status": "Unknown"
})

print("After .na.fill():")
df_filled.show()

After .na.fill():
+---+-----+--------+
| id| name|  status|
+---+-----+--------+
|  1|Alice|  Active|
|  2|  Bob| Unknown|
|  3| NULL|Inactive|
|  4|Diana| Unknown|
+---+-----+--------+



In [0]:
# Remove rows where name is null

df_dropped = df.na.drop(subset=["name"])

print("After .na.drop():")
df_dropped.show()

After .na.drop():
+---+-----+------+
| id| name|status|
+---+-----+------+
|  1|Alice|Active|
|  2|  Bob|  NULL|
|  4|Diana|  NULL|
+---+-----+------+




### Conclusion

`.na.drop()` reduces dataset size by removing incomplete records, while `.na.fill()` retains records by replacing missing values with defaults. The appropriate choice depends on business requirements and data quality rules.




## Q6. Count Records Per City, Only Where Count > 100

This task requires grouping records by city, counting the number of records in each city, and then filtering the results to keep only cities with more than 100 records.

In SQL terminology, filtering aggregated results is performed using the **HAVING** clause. In Spark's DataFrame API, the equivalent operation is applying `.filter()` after the aggregation step.

### Processing Steps

1. Group records by `city`.
2. Count the number of records in each group.
3. Filter cities where the count is greater than 100.
4. Sort the results in descending order of record count.


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# Generate sample records
data = []

# Jaipur -> 120 records
for i in range(120):
    data.append(("Jaipur",))

# Delhi -> 150 records
for i in range(150):
    data.append(("Delhi",))

# Mumbai -> 80 records
for i in range(80):
    data.append(("Mumbai",))

df = spark.createDataFrame(data, ["city"])

print("Total Records:", df.count())

Total Records: 350


In [0]:
from pyspark.sql import functions as F

city_counts = (
    df
    .groupBy("city")
    .agg(F.count("*").alias("record_count"))
    .filter(F.col("record_count") > 100)
    .orderBy(F.desc("record_count"))
)

city_counts.show()

+------+------------+
|  city|record_count|
+------+------------+
| Delhi|         150|
|Jaipur|         120|
+------+------------+



%md

## Q7. How Immutability of Spark DataFrames Affects Data Cleaning

Spark DataFrames are **immutable**, meaning that once a DataFrame is created, its data and schema cannot be modified directly.

Any transformation such as:

* `drop()`
* `filter()`
* `withColumn()`
* `withColumnRenamed()`
* `na.fill()`

does not change the original DataFrame. Instead, Spark creates and returns a **new DataFrame** containing the requested changes.

### Why Immutability Matters

Immutability provides several advantages:

| Benefit            | Explanation                                             |
| ------------------ | ------------------------------------------------------- |
| Data Safety        | Original data remains unchanged                         |
| Reproducibility    | Cleaning steps can be rerun consistently                |
| Debugging          | Easy to compare raw and cleaned datasets                |
| Fault Tolerance    | Spark can recompute lost partitions                     |
| Query Optimization | Enables Spark's Catalyst Optimizer and DAG optimization |

### Incorrect Approach

Many beginners assume transformations modify the original DataFrame.

```python
df.drop("unwanted_column")
df.withColumnRenamed("old_name", "new_name")
```

The original DataFrame remains unchanged because the returned DataFrames are not assigned to a variable.

### Correct Approach

Always capture the returned DataFrame.

```python
df_clean = df.drop("unwanted_column")
df_clean = df_clean.withColumnRenamed("old_name", "new_name")
```

### Production-Style Data Cleaning

In real-world ETL pipelines, the raw DataFrame is preserved while a separate cleaned DataFrame is created through chained transformations.

This approach improves maintainability, debugging, and auditability of data pipelines.


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    (1, "Alice", 5000, "Active"),
    (2, "Bob", -1000, None),
    (3, "Charlie", 7000, "Inactive")
]

columns = ["id", "cust_nm", "amt", "status"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show()

Original DataFrame:
+---+-------+-----+--------+
| id|cust_nm|  amt|  status|
+---+-------+-----+--------+
|  1|  Alice| 5000|  Active|
|  2|    Bob|-1000|    NULL|
|  3|Charlie| 7000|Inactive|
+---+-------+-----+--------+



In [0]:
# Transformation without assignment

df.drop("status")

print("Original DataFrame after drop():")
df.show()

Original DataFrame after drop():
+---+-------+-----+--------+
| id|cust_nm|  amt|  status|
+---+-------+-----+--------+
|  1|  Alice| 5000|  Active|
|  2|    Bob|-1000|    NULL|
|  3|Charlie| 7000|Inactive|
+---+-------+-----+--------+



The status column still exists because the original DataFrame was not modified.

In [0]:
df_clean = df.drop("status")

print("Cleaned DataFrame:")
df_clean.show()

Cleaned DataFrame:
+---+-------+-----+
| id|cust_nm|  amt|
+---+-------+-----+
|  1|  Alice| 5000|
|  2|    Bob|-1000|
|  3|Charlie| 7000|
+---+-------+-----+



In [0]:
from pyspark.sql import functions as F

df_clean = (
    df
    .withColumnRenamed("cust_nm", "customer_name")
    .withColumnRenamed("amt", "amount")
    .filter(F.col("amount") > 0)
    .na.fill({"status": "Unknown"})
)

print("Final Cleaned DataFrame:")
df_clean.show()

Final Cleaned DataFrame:
+---+-------------+------+--------+
| id|customer_name|amount|  status|
+---+-------------+------+--------+
|  1|        Alice|  5000|  Active|
|  3|      Charlie|  7000|Inactive|
+---+-------------+------+--------+



## Q8. Filter Rows Where Age is Between 18–30 and Subscription is 'Premium'

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Sample Data
data = [
    (1, "Alice", 22, "Premium"),
    (2, "Bob", 17, "Premium"),
    (3, "Charlie", 28, "Basic"),
    (4, "Diana", 25, "Premium"),
    (5, "Ethan", 31, "Premium"),
    (6, "Fiona", 19, "Premium")
]

columns = ["user_id", "name", "age", "subscription"]

df = spark.createDataFrame(data, columns)


df_filtered = df.filter(
    F.col("age").between(18, 30) &
    (F.col("subscription") == "Premium")
)

df_filtered.show()

+-------+-----+---+------------+
|user_id| name|age|subscription|
+-------+-----+---+------------+
|      1|Alice| 22|     Premium|
|      4|Diana| 25|     Premium|
|      6|Fiona| 19|     Premium|
+-------+-----+---+------------+




## Q9. Why Handle Null Values Before Mathematical Aggregations Like `sum()` or `avg()`?

Null values can significantly affect the results of mathematical aggregations. Therefore, it is important to understand how Apache Spark handles nulls before performing calculations such as `sum()`, `avg()`, `min()`, or `max()`.

### Spark's Default Behavior

Spark follows SQL semantics:

* `sum()` ignores null values.
* `avg()` ignores null values.
* Aggregations are calculated only on non-null records.

While this behavior prevents errors, it may lead to misleading results if null values have a business meaning.

### Example Scenario

Suppose an e-commerce dataset contains order prices:

| order_id | total_price |
| -------- | ----------- |
| 1        | 500         |
| 2        | null        |
| 3        | 300         |
| 4        | null        |
| 5        | 200         |

Spark computes:

* Sum = 1000
* Average = 1000 / 3 = 333.33

The average is calculated using only the three non-null values.

### Why This Can Be Dangerous

The interpretation of null values depends on business requirements.

**Case 1: Null Means Missing Data**

* Ignore null values
* Average = 333.33

**Case 2: Null Means Free Item (Price = 0)**

* Replace null values with 0
* Average = 1000 / 5 = 200

Both calculations are mathematically correct, but they answer different business questions.

### Best Practice

Always decide what null values represent before performing aggregations.

Possible strategies include:

* Replace nulls using `.na.fill()`
* Remove records using `.na.drop()`
* Keep nulls if business logic requires ignoring them

Making this decision explicitly helps avoid misleading analytics and reporting results.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

data = [
    (1, 500.0),
    (2, None),
    (3, 300.0),
    (4, None),
    (5, 200.0)
]

df = spark.createDataFrame(data, ["order_id", "total_price"])

print("Original Data:")
df.show()

Original Data:
+--------+-----------+
|order_id|total_price|
+--------+-----------+
|       1|      500.0|
|       2|       NULL|
|       3|      300.0|
|       4|       NULL|
|       5|      200.0|
+--------+-----------+



In [0]:
print("Sum and Average without handling nulls:")

df.agg(
    F.sum("total_price").alias("total_sales"),
    F.avg("total_price").alias("average_price")
).show()

Sum and Average without handling nulls:
+-----------+-----------------+
|total_sales|    average_price|
+-----------+-----------------+
|     1000.0|333.3333333333333|
+-----------+-----------------+



In [0]:
df_filled = df.na.fill({
    "total_price": 0
})

print("After replacing nulls with 0:")

df_filled.agg(
    F.sum("total_price").alias("total_sales"),
    F.avg("total_price").alias("average_price")
).show()

After replacing nulls with 0:
+-----------+-------------+
|total_sales|average_price|
+-----------+-------------+
|     1000.0|        200.0|
+-----------+-------------+



In [0]:
df_dropped = df.na.drop(subset=["total_price"])

print("After dropping null records:")

df_dropped.agg(
    F.sum("total_price").alias("total_sales"),
    F.avg("total_price").alias("average_price")
).show()

After dropping null records:
+-----------+-----------------+
|total_sales|    average_price|
+-----------+-----------------+
|     1000.0|333.3333333333333|
+-----------+-----------------+



## Q10. Cast raw_timestamp to TimestampType and Rename to event_time 

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.getOrCreate()

# Sample dataset
data = [
    (1, "2024-01-15 10:30:00"),
    (2, "2024-01-16 14:45:30"),
    (3, "2024-01-17 09:15:10")
]

columns = ["event_id", "raw_timestamp"]

df = spark.createDataFrame(data, columns)

print("Original DataFrame:")
df.show(truncate=False)
df.printSchema()

# Cast raw_timestamp to TimestampType and rename to event_time
df_transformed = (
    df.withColumn("event_time", col("raw_timestamp").cast("timestamp"))
      .drop("raw_timestamp")
)

print("Transformed DataFrame:")
df_transformed.show(truncate=False)
df_transformed.printSchema()

Original DataFrame:
+--------+-------------------+
|event_id|raw_timestamp      |
+--------+-------------------+
|1       |2024-01-15 10:30:00|
|2       |2024-01-16 14:45:30|
|3       |2024-01-17 09:15:10|
+--------+-------------------+

root
 |-- event_id: long (nullable = true)
 |-- raw_timestamp: string (nullable = true)

Transformed DataFrame:
+--------+-------------------+
|event_id|event_time         |
+--------+-------------------+
|1       |2024-01-15 10:30:00|
|2       |2024-01-16 14:45:30|
|3       |2024-01-17 09:15:10|
+--------+-------------------+

root
 |-- event_id: long (nullable = true)
 |-- event_time: timestamp (nullable = true)



%md

## Q11. The Shuffle Process During Grouping — Why It Is a Wide Transformation

Understanding **shuffle** is essential for optimizing Spark applications because it is one of the most expensive operations in distributed processing.

### Narrow vs. Wide Transformations

| Transformation Type | Definition                                                  | Examples                            | Shuffle Required? |
| ------------------- | ----------------------------------------------------------- | ----------------------------------- | ----------------- |
| Narrow              | Each output partition depends on only one input partition   | `map()`, `filter()`, `select()`     | No                |
| Wide                | An output partition may depend on multiple input partitions | `groupBy()`, `join()`, `distinct()` | Yes               |

### What Happens During a `groupBy()` Operation?

Suppose records belonging to the same city are distributed across multiple partitions.

When Spark executes:

```python
df.groupBy("city").count()
```

it must ensure that all records for a particular city are brought together into the same partition before aggregation can occur.

The process is:

1. Data is distributed across multiple partitions.
2. Spark computes a hash of the grouping key.
3. Records are redistributed across executors based on the hash value.
4. Data is transferred across the network.
5. Aggregation occurs after all matching keys are colocated.

This redistribution of data is called a **shuffle**.

### Why Is Shuffle Expensive?

* Network I/O is required.
* Intermediate data may be written to disk.
* Executors must wait for shuffle completion.
* Data skew can create bottlenecks.
* Shuffle increases overall job execution time.

### Tuning Shuffle Partitions

Spark uses the configuration:

```python
spark.conf.set("spark.sql.shuffle.partitions", "50")
```

to determine the number of output shuffle partitions.

Choosing an appropriate value can significantly improve performance.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Sample dataset
data = [
    ("Mumbai",),
    ("Delhi",),
    ("Mumbai",),
    ("Jaipur",),
    ("Delhi",),
    ("Mumbai",),
    ("Pune",),
    ("Jaipur",),
    ("Mumbai",),
    ("Delhi",)
]

df = spark.createDataFrame(data, ["city"])

print("Original Data")
df.show()


Original Data
+------+
|  city|
+------+
|Mumbai|
| Delhi|
|Mumbai|
|Jaipur|
| Delhi|
|Mumbai|
|  Pune|
|Jaipur|
|Mumbai|
| Delhi|
+------+



In [0]:
# groupBy triggers shuffle
result = (
    df.groupBy("city")
      .count()
      .orderBy(F.desc("count"))
)

print("Grouped Result")
result.show()

Grouped Result
+------+-----+
|  city|count|
+------+-----+
|Mumbai|    4|
| Delhi|    3|
|Jaipur|    2|
|  Pune|    1|
+------+-----+



In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim

spark = SparkSession.builder.getOrCreate()

# Sample dataset
data = [
    (1, "harshit", "harshit@gmail.com"),
    (2, "", "bob@gmail.com"),              # empty username
    (3, "charlie", None),                  # null email
    (4, "diana", "diana@gmail.com"),
    (5, "   ", "eve@gmail.com"),           # whitespace username
]

columns = ["user_id", "username", "email"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show(truncate=False)


Original Data:
+-------+--------+-----------------+
|user_id|username|email            |
+-------+--------+-----------------+
|1      |harshit |harshit@gmail.com|
|2      |        |bob@gmail.com    |
|3      |charlie |NULL             |
|4      |diana   |diana@gmail.com  |
|5      |        |eve@gmail.com    |
+-------+--------+-----------------+



In [0]:
# Remove rows where email is NULL OR username is empty/blank
print("Remove rows where email is NULL OR username is empty/blank")
df_clean = df.filter(
    col("email").isNotNull() &
    (trim(col("username")) != "")
)

print("Cleaned Data:")
df_clean.show(truncate=False)

Remove rows where email is NULL OR username is empty/blank
Cleaned Data:
+-------+--------+-----------------+
|user_id|username|email            |
+-------+--------+-----------------+
|1      |harshit |harshit@gmail.com|
|4      |diana   |diana@gmail.com  |
+-------+--------+-----------------+



## Q13. Using .agg() to Calculate min, max, and mean of the price Column

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Sample dataset
data = [
    (1, 500.0),
    (2, 750.0),
    (3, 300.0),
    (4, 1200.0),
    (5, 950.0)
]

columns = ["product_id", "price"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()

# Calculate min, max, and mean of price column
price_stats = df.agg(
    F.min("price").alias("min_price"),
    F.max("price").alias("max_price"),
    F.mean("price").alias("avg_price")
)

print("Price Statistics:")
price_stats.show()

Original Data:
+----------+------+
|product_id| price|
+----------+------+
|         1| 500.0|
|         2| 750.0|
|         3| 300.0|
|         4|1200.0|
|         5| 950.0|
+----------+------+

Price Statistics:
+---------+---------+---------+
|min_price|max_price|avg_price|
+---------+---------+---------+
|    300.0|   1200.0|    740.0|
+---------+---------+---------+



%md

## Q14. Risk of Using `inferSchema=true` with Messy or Inconsistent Date Formats

Schema inference (`inferSchema=true`) is convenient because Spark automatically determines column data types. However, it can be risky when date columns contain inconsistent formats.

### How Schema Inference Works

When `inferSchema=true` is enabled:

1. Spark samples a portion of the dataset.
2. It determines the most likely data type for each column.
3. The inferred schema is applied to the entire dataset.

### Potential Problem

Consider a date column containing multiple formats:

* `2024-01-15` (YYYY-MM-DD)
* `15/01/2024` (DD/MM/YYYY)
* `01-15-2024` (MM-DD-YYYY)
* `Jan 15 2024` (Text Format)

If Spark infers the column as a DateType using only the first format, rows containing other formats may fail to parse and become null values.

This can lead to:

* Data loss
* Incorrect aggregations
* Inaccurate reporting
* Difficult debugging

### Best Practice

In production pipelines:

* Read date columns as strings.
* Define schemas explicitly.
* Parse dates using known formats.
* Log records that fail parsing.

This approach provides greater control, transparency, and data quality.


In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

data = [
    ("1001", "2024-01-15", "500"),
    ("1002", "15/01/2024", "300"),
    ("1003", "01-15-2024", "200"),
    ("1004", "Jan 15 2024", "400")
]

columns = ["order_id", "event_date", "amount"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show(truncate=False)

Original Data:
+--------+-----------+------+
|order_id|event_date |amount|
+--------+-----------+------+
|1001    |2024-01-15 |500   |
|1002    |15/01/2024 |300   |
|1003    |01-15-2024 |200   |
|1004    |Jan 15 2024|400   |
+--------+-----------+------+



In [0]:
from pyspark.sql import functions as F

df_clean = (
    df.withColumn(
        "event_date_parsed",
        F.coalesce(
            F.try_to_timestamp("event_date", F.lit("yyyy-MM-dd")),
            F.try_to_timestamp("event_date", F.lit("dd/MM/yyyy")),
            F.try_to_timestamp("event_date", F.lit("MM-dd-yyyy")),
        )
    )
)
df_clean.show(truncate=False)

+--------+-----------+------+-------------------+
|order_id|event_date |amount|event_date_parsed  |
+--------+-----------+------+-------------------+
|1001    |2024-01-15 |500   |2024-01-15 00:00:00|
|1002    |15/01/2024 |300   |2024-01-15 00:00:00|
|1003    |01-15-2024 |200   |2024-01-15 00:00:00|
|1004    |Jan 15 2024|400   |NULL               |
+--------+-----------+------+-------------------+



In [0]:
invalid_dates = df_clean.filter(
    F.col("event_date_parsed").isNull()
)

print("Rows with invalid date formats:")
invalid_dates.show(truncate=False)

Rows with invalid date formats:
+--------+-----------+------+-----------------+
|order_id|event_date |amount|event_date_parsed|
+--------+-----------+------+-----------------+
|1004    |Jan 15 2024|400   |NULL             |
+--------+-----------+------+-----------------+



## Q15. Final Processing Pipeline: Deduplicate → Fill Null Prices → Group by store_id

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.getOrCreate()

# Sample raw data
raw_data = [
    ("S001", "P101", 150.0),
    ("S001", "P101", 150.0),  # duplicate
    ("S002", "P201", None),   # null price
    ("S002", "P202", 200.0),
    ("S003", "P301", 350.0),
    ("S001", "P102", None)    # null price
]

df_raw = spark.createDataFrame(
    raw_data,
    ["store_id", "product_id", "price"]
)

# Final Processing Pipeline:
# 1. Deduplicate
# 2. Fill null prices with 0
# 3. Group by store_id and calculate revenue metrics

df_result = (
    df_raw
    .dropDuplicates()
    .na.fill({"price": 0})
    .groupBy("store_id")
    .agg(
        F.sum("price").alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.avg("price").alias("avg_price_per_txn")
    )
    .orderBy("store_id")
)

df_result.show()

+--------+-------------+-----------------+-----------------+
|store_id|total_revenue|transaction_count|avg_price_per_txn|
+--------+-------------+-----------------+-----------------+
|    S001|        150.0|                2|             75.0|
|    S002|        200.0|                2|            100.0|
|    S003|        350.0|                1|            350.0|
+--------+-------------+-----------------+-----------------+

